In [18]:
import pandas as pd
from pipepline_utils import load_all_zips, AccelPipeline
from sklearn.pipeline import  Pipeline

from modeling_utils  import ActivityModeler
data = load_all_zips('/home/rajesh/work/acclerometer_project/zip_data')
pipeline = AccelPipeline(data)
pipeline.convert_to_gravity()
pipeline.calc_odba()
pipeline.calc_vedba()




--- Converting to Gravity Units & Calculating ENMO ---
--- Calculating ODBA ---
--- Calculating VeDBA ---


,UTCdate,local_ts,x,y,z,subject,behavioral_category,behavior_type,x_g,y_g,z_g,mag,enmo,odba,vedba
0,2025-07-22 22:21:35,2025-07-22 17:21:35,-16536.0,-2196.0,-1392.0,500,Resting,START,-1.009277,-0.134033,-0.084961,1.021677,0.021677,0.049805,0.034024
1,2025-07-22 22:21:35,2025-07-22 17:21:35,-16536.0,-2196.0,-1392.0,500,Resting,START,-1.009277,-0.134033,-0.084961,1.021677,0.021677,0.024902,0.017012
2,2025-07-22 22:21:35,2025-07-22 17:21:35,-16608.0,-1584.0,-1968.0,500,Resting,START,-1.013672,-0.096680,-0.120117,1.025332,0.025332,0.042236,0.028102
3,2025-07-22 22:21:35,2025-07-22 17:21:35,-16476.0,-972.0,-1548.0,500,Resting,START,-1.005615,-0.059326,-0.094482,1.011785,0.011785,0.051819,0.036953
4,2025-07-22 22:21:35,2025-07-22 17:21:35,-16608.0,-2292.0,-1104.0,500,Resting,START,-1.013672,-0.139893,-0.067383,1.025496,0.025496,0.037646,0.026934
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2450928,2025-07-30 17:28:15,2025-07-30 12:28:15,-15156.0,-3036.0,-5628.0,1144,Resting,START,-0.925049,-0.185303,-0.343506,1.004016,0.004016,0.037231,0.025439
2450929,2025-07-30 17:28:15,2025-07-30 12:28:15,-14628.0,-3252.0,-5868.0,1144,Resting,START,-0.892822,-0.198486,-0.358154,0.982244,0.000000,0.031913,0.021805
2450930,2025-07-30 17:28:15,2025-07-30 12:28:15,-15036.0,-3036.0,-5652.0,1144,Resting,START,-0.917725,-0.185303,-0.344971,0.997778,0.000000,0.029938,0.022365
2450931,2025-07-30 17:28:15,2025-07-30 12:28:15,-15180.0,-2916.0,-5748.0,1144,Resting,START,-0.926514,-0.177979,-0.350830,1.006571,0.006571,0.031169,0.023724


In [ ]:

time_steps=[100]
temp_results = []
resampled_map = {}

for t in time_steps:
    df_ready = pipeline.resample_data(interval_seconds=t)

    # skip if no data or no target
    if df_ready is None or df_ready.empty or 'behavioral_category' not in df_ready.columns:
        continue
    df_ready= df_ready.sample(frac=0.01)
    modeler = ActivityModeler(df_ready, target_col='behavioral_category')
    results = modeler.run_optuna_experiments()

    # support (df, path) return or df only
    if isinstance(results, tuple):
        results_df = results[0]
    else:
        results_df = results

    if results_df is None or results_df.empty:
        continue

    results_df = results_df.copy()
    results_df['time'] = t
    temp_results.append(results_df)
    resampled_map[t] = df_ready

if not temp_results:
    return None

final_df = pd.concat(temp_results, ignore_index=True)

# pick best by highest F1
best_row = final_df.sort_values('F1_Score', ascending=False).iloc[0]

best_time = best_row['time']
best_algo = best_row['Algorithm']

# parse params and features (they may be stored as strings)
try:
    best_params = ast.literal_eval(best_row['Best_Params'])
except Exception:
    best_params = {}

try:
    best_features = ast.literal_eval(best_row['Features_Used'])
except Exception:
    best_features = best_row['Features_Used']

# get the resampled dataframe for the best time
df_best = resampled_map.get(best_time)
if df_best is None:
     None

X_full = df_best[best_features]
y_full = LabelEncoder().fit_transform(df_best['behavioral_category'])

# reconstruct model
if best_algo == "XGBoost":
    model = xgb.XGBClassifier(**best_params, use_label_encoder=False, eval_metric='mlogloss', random_state=42)
elif best_algo == "RandomForest":
    model = RandomForestClassifier(**best_params, random_state=42)
elif best_algo == "SVM (RBF)":
    model = SVC(**best_params, kernel='rbf', random_state=42)
elif best_algo == "KNN":
    model = KNeighborsClassifier(**best_params)
elif best_algo == "LogisticReg":
    model = LogisticRegression(**best_params, solver='lbfgs', max_iter=2000, random_state=42)
else:
     None

pipeline_obj = Pipeline([('scaler', StandardScaler()), ('model', model)])
pipeline_obj.fit(X_full, y_full)

model_dict= {
    'model': pipeline_obj,
    'time': best_time,
    'algorithm': best_algo,
    'best_params': best_params,
    'features': best_features,
    'f1_score': best_row.get('F1_Score', None)
}

--- Resampling data to 100 second windows ---
Starting Optuna Tuning on 21 rows.
Optimizing 8 Feature Sets x 5 Models...
------------------------------------------------------------
 Tuning XGBoost | Features: Indiv: Raw Accel...


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, whi

 Tuning RandomForest | Features: Indiv: Raw Accel...


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, whi

 Tuning SVM (RBF) | Features: Indiv: Raw Accel...
 Tuning KNN | Features: Indiv: Raw Accel...
 Tuning LogisticReg | Features: Indiv: Raw Accel...


...

 Tuning SVM (RBF) | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...
 Tuning KNN | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
            

 Tuning LogisticReg | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


SyntaxError: 'return' outside function (3098671809.py, line 30)

In [26]:
best_features

NameError: name 'best_features' is not defined

In [15]:
trained_model= get_best_classical_model(pipeline=pipeline, save_dir='"/home/rajesh/work/acclerometer_project',
                                                                time_steps=[100]                )


--- Resampling data to 100 second windows ---
Starting Optuna Tuning on 21 rows.
Optimizing 8 Feature Sets x 5 Models...
------------------------------------------------------------
 Tuning XGBoost | Features: Indiv: Raw Accel...


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [17:16:02] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Tuning RandomForest | Features: Indiv: Raw Accel...
 Tuning SVM (RBF) | Features: Indiv: Raw Accel...
 Tuning KNN | Features: Indiv: Raw Accel...


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/utils/_response.py", line 214, in _get_response_values
    y_pred = prediction_method(X)
             ^^^^^^^^^^^^^^^^^^^^
  File "/

 Tuning LogisticReg | Features: Indiv: Raw Accel...
 Tuning XGBoost | Features: Indiv: ODBA...


...

/home/rajesh/work/basic_stat/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [17:18:03] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Tuning RandomForest | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/utils/_response.py", line 214, in _get_response_values
    y_pred = prediction_method(X)
             ^^^^^^^^^^^^^^^^^^^^
  File "/

 Tuning SVM (RBF) | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...
 Tuning KNN | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...
 Tuning LogisticReg | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...


KeyError: "['x_g', 'y_g', 'z_g']"